# 🚀 Library Management System

**Lumexa Python Foundations · Course 2: Problem Solving & Program Design · Lesson 8**

| | |
|---|---|
| **Project** | Library Management System (portfolio project 6 of 9) |
| **You will practise** | classes and objects, attributes and methods, `__init__`, `self`, encapsulation, dictionaries, JSON storage, error handling |
| **Time** | about 75 minutes |

This is your **Course 2 capstone**. You will design three classes that work together (`Book`, `Member` and `Library`) to manage books, borrowers and borrowing records, protect the rules of the library with validation and save everything to a file.

## 🧭 How to use this notebook

1. **Keep your own copy.** Choose **File → Save a copy in Drive** first, so your work is saved in *your* Google Drive.
2. **Run the cells in order**, from the top, with **Shift + Enter** (or **Runtime → Run all**). Read the text above each cell before you run it.
3. Cells marked **✏️ Your turn** are for you to change or complete. They are safe to run before you finish them.
4. If something goes wrong, use **Runtime → Restart session and run all**. A notebook you can run from the top is a notebook you can trust.
5. **Demo mode:** by default the notebook "types" the answers for you, so *Run all* works without stopping. To play yourself, set `INTERACTIVE = True` in the setup cell and run it again.

In [ ]:
# ⚙️ Setup — run this cell first
# INTERACTIVE = False -> demo mode: the notebook "types" scripted answers, so Run all works.
# INTERACTIVE = True  -> you type the answers yourself.
INTERACTIVE = False

demo_script = iter([])      # the scripted answers for demo mode

def ask(prompt):
    """Works like input(). In demo mode the next scripted answer is 'typed' for you."""
    if INTERACTIVE:
        return input(prompt)
    answer = next(demo_script, "")
    print(f"{prompt}{answer}")
    return answer

import json
from datetime import date, timedelta
from pathlib import Path

LOAN_DAYS = 14          # how long a book can be borrowed
MAX_BOOKS = 3           # how many books one member may have at once
FEE_PER_DAY = 5         # late fee for each day after the due date (৳)

print("Setup complete. Interactive mode is", "ON" if INTERACTIVE else "OFF (demo mode)")

## Part 1 · Design first: what are the "things"?

Before writing code, ask: *what are the things in my program, what do they **know**, and what can they **do**?*

| Class | Knows (attributes) | Can do (methods) |
|---|---|---|
| **Book** | ISBN, title, author, whether it is available | describe itself, convert itself to a dictionary |
| **Member** | member id, name, the ISBNs of the books they have borrowed | say whether they may borrow another book |
| **Library** | all the books, all the members, all the borrowing records | add a book, register a member, borrow, return, search, report, save, load |

Each class has **one clear job**. The `Library` is the manager: it owns books and members and enforces the rules.

## Part 2 · The `Book` and `Member` classes

`__init__()` is the **constructor**: it runs when an object is created and sets up its attributes. `self` means "this object". `__str__()` says how an object looks when you `print()` it.

In [ ]:
class Book:
    """One book in the library."""

    def __init__(self, isbn, title, author, available=True):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.available = available

    def __str__(self):
        status = "available" if self.available else "ON LOAN"
        return f"[{self.isbn}] {self.title} by {self.author} - {status}"

    def to_dict(self):
        return {"isbn": self.isbn, "title": self.title, "author": self.author, "available": self.available}


class Member:
    """A library member who may borrow up to MAX_BOOKS books at a time."""

    def __init__(self, member_id, name, borrowed=None):
        self.member_id = member_id
        self.name = name
        self.borrowed = borrowed if borrowed is not None else []     # the ISBNs on loan

    def can_borrow(self):
        return len(self.borrowed) < MAX_BOOKS

    def __str__(self):
        return f"{self.name} ({self.member_id}) has {len(self.borrowed)} book(s) on loan"

    def to_dict(self):
        return {"member_id": self.member_id, "name": self.name, "borrowed": self.borrowed}


book = Book("B001", "Python for Space Cadets", "Nova Reed")
member = Member("M1", "Asha")
print(book)
print(member)
print("Can Asha borrow?", member.can_borrow())

## Part 3 · The `Library` class

The `Library` holds **dictionaries** of books and members (looked up by their id) and a **list of borrowing records**. Each record remembers who borrowed what, when it is due and when it came back.

Notice how the rules are protected **inside the methods**. When something is not allowed, the method **raises a `ValueError`** with a clear message instead of quietly doing the wrong thing. This is *encapsulation*: nobody can break the rules by changing the data from outside.

> Dates are passed in as `today`, so that we can test the late-fee rule with dates we choose. If you leave it out, the real date of today is used.

In [ ]:
class Library:
    """Manages books, members and borrowing records."""

    def __init__(self):
        self.books = {}        # isbn -> Book
        self.members = {}      # member_id -> Member
        self.records = []      # one dictionary for every loan

    # ----- adding things -----
    def add_book(self, isbn, title, author):
        if isbn in self.books:
            raise ValueError(f"A book with ISBN {isbn} already exists.")
        self.books[isbn] = Book(isbn, title, author)

    def register_member(self, member_id, name):
        if member_id in self.members:
            raise ValueError(f"Member id {member_id} is already registered.")
        self.members[member_id] = Member(member_id, name)

    # ----- borrowing and returning -----
    def borrow_book(self, member_id, isbn, today=None):
        today = today or date.today()
        member = self.members.get(member_id)
        book = self.books.get(isbn)
        if member is None:
            raise ValueError(f"Unknown member: {member_id}")
        if book is None:
            raise ValueError(f"Unknown book: {isbn}")
        if not book.available:
            raise ValueError(f"'{book.title}' is already on loan.")
        if not member.can_borrow():
            raise ValueError(f"{member.name} has reached the limit of {MAX_BOOKS} books.")

        book.available = False
        member.borrowed.append(isbn)
        record = {
            "isbn": isbn,
            "member_id": member_id,
            "borrowed_on": today.isoformat(),
            "due_on": (today + timedelta(days=LOAN_DAYS)).isoformat(),
            "returned_on": None,
            "fee": 0,
        }
        self.records.append(record)
        return record

    def find_open_record(self, member_id, isbn):
        """The loan record of a book that has not been returned yet (or None)."""
        for record in self.records:
            if record["member_id"] == member_id and record["isbn"] == isbn and record["returned_on"] is None:
                return record
        return None

    def return_book(self, member_id, isbn, today=None):
        """Return a book and give back the late fee (0 if it is on time)."""
        today = today or date.today()
        record = self.find_open_record(member_id, isbn)
        if record is None:
            raise ValueError(f"Member {member_id} does not have book {isbn}.")
        days_late = max(0, (today - date.fromisoformat(record["due_on"])).days)
        record["returned_on"] = today.isoformat()
        record["fee"] = days_late * FEE_PER_DAY
        self.books[isbn].available = True
        self.members[member_id].borrowed.remove(isbn)
        return record["fee"]

    # ----- finding things out -----
    def search(self, text):
        """Books whose title or author contains the text (capital letters are ignored)."""
        text = text.lower()
        return [b for b in self.books.values() if text in b.title.lower() or text in b.author.lower()]

    def overdue(self, today=None):
        today = today or date.today()
        return [r for r in self.records if r["returned_on"] is None and date.fromisoformat(r["due_on"]) < today]

    def report(self):
        on_loan = [b for b in self.books.values() if not b.available]
        print(f"Books: {len(self.books)} ({len(on_loan)} on loan) | Members: {len(self.members)} | Loans recorded: {len(self.records)}")

    # ----- saving and loading -----
    def save(self, path):
        data = {
            "books": [b.to_dict() for b in self.books.values()],
            "members": [m.to_dict() for m in self.members.values()],
            "records": self.records,
        }
        with open(path, "w") as file:
            json.dump(data, file, indent=2)

    def load(self, path):
        with open(path) as file:
            data = json.load(file)
        self.books = {}
        for b in data["books"]:
            self.books[b["isbn"]] = Book(b["isbn"], b["title"], b["author"], b["available"])
        self.members = {}
        for m in data["members"]:
            self.members[m["member_id"]] = Member(m["member_id"], m["name"], m["borrowed"])
        self.records = data["records"]

print("The Library class is ready.")

## Part 4 · Stocking the library

We create one `Library` **object** and add books and members to it.

In [ ]:
library = Library()
library.add_book("B001", "Python for Space Cadets", "Nova Reed")
library.add_book("B002", "The Red Planet Handbook", "Orion Hale")
library.add_book("B003", "Robots and Rovers", "Lyra Chen")
library.add_book("B004", "Learning to Code with Games", "Kai Moreno")
library.add_book("B005", "Stars, Moons and Planets", "Mia Okafor")
library.register_member("M1", "Asha")
library.register_member("M2", "Rafi")

library.report()
for book in library.books.values():
    print("  ", book)

## Part 5 · Borrowing, returning and the rules

We run a **script of actions**, including several that should be **refused**: borrowing a book that is already out, going over the limit of three books, an unknown member and an unknown book. Each refusal is a `ValueError` that we catch with `try` and `except` and show as a friendly message.

The dates are chosen so that one book is returned on time and one is returned five days late.

In [ ]:
def perform(library, step):
    """Carry out one (action, member, book, date) step and print what happened."""
    action, member_id, isbn, day = step
    try:
        if action == "borrow":
            record = library.borrow_book(member_id, isbn, today=day)
            print(f"✅ {day}  {member_id} borrowed {isbn} (due {record['due_on']})")
        else:
            fee = library.return_book(member_id, isbn, today=day)
            note = f" - late fee: ৳{fee}" if fee else " - on time"
            print(f"✅ {day}  {member_id} returned {isbn}{note}")
    except ValueError as error:
        print(f"❌ {day}  {action} {member_id}/{isbn}: {error}")

steps = [
    ("borrow", "M1", "B001", date(2030, 3, 1)),
    ("borrow", "M1", "B002", date(2030, 3, 1)),
    ("borrow", "M1", "B003", date(2030, 3, 1)),
    ("borrow", "M1", "B004", date(2030, 3, 2)),    # refused: Asha already has 3 books
    ("borrow", "M2", "B001", date(2030, 3, 2)),    # refused: B001 is already on loan
    ("borrow", "M9", "B005", date(2030, 3, 2)),    # refused: no such member
    ("borrow", "M2", "B099", date(2030, 3, 2)),    # refused: no such book
    ("return", "M1", "B001", date(2030, 3, 10)),   # on time (due 3/15)
    ("return", "M1", "B002", date(2030, 3, 20)),   # 5 days late
    ("return", "M2", "B005", date(2030, 3, 20)),   # refused: Rafi never borrowed it
    ("borrow", "M2", "B001", date(2030, 3, 21)),   # now it is free again
]
for step in steps:
    perform(library, step)

print()
library.report()

## Part 6 · Searching and the overdue report

In [ ]:
for text in ["python", "NOVA", "planet", "dragon"]:
    found = library.search(text)
    print(f"Search for '{text}': {len(found)} book(s)")
    for book in found:
        print("    ", book)

print()
check_date = date(2030, 3, 25)
late = library.overdue(today=check_date)
print(f"Overdue on {check_date}: {len(late)} book(s)")
for record in late:
    days = (check_date - date.fromisoformat(record["due_on"])).days
    print(f"   {record['isbn']} borrowed by {record['member_id']}, {days} day(s) overdue (fee so far: ৳{days * FEE_PER_DAY})")

## Part 7 · Saving and loading

A real library cannot forget its books when the program stops. We save everything to a JSON file, create a **brand-new, empty** `Library` and load the file into it. If the two libraries match, the saving worked.

In [ ]:
DATA_FOLDER = Path("library_data")
DATA_FOLDER.mkdir(exist_ok=True)
LIBRARY_FILE = DATA_FOLDER / "library.json"

library.save(LIBRARY_FILE)
print("Saved to", LIBRARY_FILE)

copy = Library()
copy.load(LIBRARY_FILE)
print("Loaded library:")
copy.report()
print("Same books?   ", sorted(copy.books) == sorted(library.books))
print("Same records? ", copy.records == library.records)
print("Asha's loans:", copy.members["M1"])

## Part 8 · The menu

The `main` menu ties it all together. Every action calls a method of the `Library`, and every refusal is caught and reported. In demo mode a scripted session runs (it uses a fixed date so the results are repeatable).

In [ ]:
MENU = """
===== LIBRARY MANAGEMENT SYSTEM =====
1. Borrow a book
2. Return a book
3. Search for a book
4. Library report
5. Quit"""

def run_menu(library):
    today = None if INTERACTIVE else date(2030, 4, 1)      # a fixed date in demo mode
    while True:
        print(MENU)
        choice = ask("Choose 1-5: ").strip()
        try:
            if choice == "1":
                member_id = ask("Member id: ").strip().upper()
                isbn = ask("Book ISBN: ").strip().upper()
                record = library.borrow_book(member_id, isbn, today=today)
                print(f"  ✅ Borrowed. Please return it by {record['due_on']}.")
            elif choice == "2":
                member_id = ask("Member id: ").strip().upper()
                isbn = ask("Book ISBN: ").strip().upper()
                fee = library.return_book(member_id, isbn, today=today)
                print(f"  ✅ Returned. Late fee: ৳{fee}")
            elif choice == "3":
                for book in library.search(ask("Search for: ")):
                    print("   ", book)
            elif choice == "4":
                library.report()
            elif choice == "5":
                print("Goodbye! 👋")
                break
            else:
                print("  ⚠️  Please choose a number from 1 to 5.")
        except ValueError as error:
            print(f"  ❌ {error}")

demo_script = iter(["3", "robots", "1", "m2", "b003", "1", "m2", "b003", "2", "m2", "b003", "7", "4", "5"])
run_menu(library)

## ✏️ Your turn · Who has what?

Write `member_loans(library, member_id)`. It returns a **list of the titles** of the books that member currently has on loan (an empty list if none). Hint: a member's `borrowed` list holds ISBNs; look each one up in `library.books`.

In [ ]:
def member_loans(library, member_id):
    return None   # <- write your code here, then run the check below

mine = member_loans(library, "M1")
expected = ["Robots and Rovers"]
print("✅ member_loans works!" if mine == expected else f"⬜ member_loans is not finished yet (expected {expected}, got {mine})")

## 🏆 Stretch challenges

1. Write a `Magazine` class that works like `Book` but also has an `issue_number`. Can the library hold both?
2. Add a **`search_by_author()`** method and a method that lists every book a member has borrowed so far (not only today).
3. Add `report()` details: the most active member and the total late fees collected.
4. Write at least **eight `assert` tests** that create a small library and check borrowing, returning, the limit and searching.
5. Make the library **remember who owes fees** and refuse new loans until they are paid.

## 🪞 Reflection

Write a short **design note**: what is each class responsible for, which rule is protected where, and what would you change with more time?

*Your design note:*

## 📤 How to submit

**File → Save a copy in Drive**, then **Share → Anyone with the link can view**, and send the link to your teacher. Make sure the notebook runs from the top with **Runtime → Restart session and run all**.